# PhysicsNeMo Darcy FNO Colab Starter

이 노트북은 NVIDIA PhysicsNeMo의 Darcy FNO 예제를 Colab에서 공부하고 커스텀 실험으로 확장하기 위한 버전입니다.

핵심 목표:

1. Colab 런타임/GPU/인코딩 문제를 자동 진단합니다.
2. PhysicsNeMo 패키지와 공식 예제를 셀 단위로 확인합니다.
3. Darcy 데이터가 어떤 tensor shape과 값을 가지는지 시각화합니다.
4. FNO 모델 구조와 입력/출력 흐름을 설명합니다.
5. 공식 학습 스크립트가 깨질 때도 노트북 전체가 멈추지 않게 합니다.
6. 직접 커스텀하기 좋은 위치를 셀 단위로 보여줍니다.

공식 예제: https://github.com/NVIDIA/physicsnemo/tree/main/examples/cfd/darcy_fno

## 전체 플로우

Darcy FNO 예제는 다음 문제를 풉니다.

```text
permeability field a(x, y)  ->  Darcy solution u(x, y)
```

- 입력 `permeability`: 격자 위의 투과도/계수 field입니다.
- 정답 `darcy`: Darcy PDE를 만족하는 해 field입니다.
- 모델 `FNO`: Fourier domain에서 operator를 학습하는 neural operator입니다.
- 목표: 새로운 permeability field가 들어와도 전체 solution field를 빠르게 예측하는 surrogate model을 만드는 것입니다.

Tensor shape 기본값:

```text
permeability: [batch, channel=1, height, width]
darcy:        [batch, channel=1, height, width]
prediction:   [batch, channel=1, height, width]
```

In [ ]:
# 0. Runtime auto-fix: GPU, CUDA, UTF-8 encoding
import os
import sys
import locale
import subprocess
import importlib
from pathlib import Path

# Colab/Python UTF-8 guard. This fixes common locale/encoding errors from pip or subprocesses.
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUTF8"] = "1"
os.environ.setdefault("LANG", "C.UTF-8")
os.environ.setdefault("LC_ALL", "C.UTF-8")
try:
    locale.getpreferredencoding = lambda do_setlocale=True: "UTF-8"
except Exception as exc:
    print("UTF-8 locale patch skipped:", repr(exc))

print("python:", sys.version.split()[0])
print("preferred encoding:", locale.getpreferredencoding())

try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found. Continuing with CPU fallback.")

import torch
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else "CPU fallback"

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("device:", DEVICE)
print("gpu:", GPU_NAME)

AUTO_FIX = {
    "has_gpu": HAS_GPU,
    "device": DEVICE,
    "gpu_name": GPU_NAME,
    "encoding": locale.getpreferredencoding(),
}

if not HAS_GPU:
    print("WARNING: GPU is not available. The notebook will use tiny CPU configs and skip heavy runs when needed.")

## 저장 위치

Drive mount는 Colab 인증 상태에 따라 실패할 수 있습니다. 실패하면 `/content/physicsnemo_runs/darcy_fno` 임시 저장소를 사용합니다.

- Drive 성공: 세션 종료 후에도 checkpoint/log가 남습니다.
- Drive 실패: 실험은 계속 가능하지만 런타임 종료 시 결과가 사라질 수 있습니다.

In [ ]:
RUN_ROOT = "/content/physicsnemo_runs/darcy_fno"

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = "/content/drive/MyDrive/physicsnemo_runs/darcy_fno"
except Exception as exc:
    print("Drive mount skipped:", repr(exc))
    print("Using temporary runtime storage instead.")

Path(RUN_ROOT).mkdir(parents=True, exist_ok=True)
print("RUN_ROOT:", RUN_ROOT)

## 설치

Darcy FNO smoke test에는 먼저 최소 설치를 사용합니다. `nn-extras`는 무겁고 Colab 런타임 재시작을 유발할 수 있어서, 필요한 실험에서만 나중에 추가합니다.

설치 셀이 실패하면 다음 셀의 import 진단에서 어떤 패키지가 빠졌는지 확인합니다.

In [ ]:
!pip install -U pip
!pip install nvidia-physicsnemo hydra-core omegaconf termcolor warp-lang matplotlib

In [ ]:
# 설치/import 진단
REQUIRED_IMPORTS = {
    "physicsnemo": "physicsnemo",
    "hydra": "hydra",
    "omegaconf": "omegaconf",
    "warp": "warp",
    "matplotlib": "matplotlib",
    "torch": "torch",
}

missing = []
for label, module_name in REQUIRED_IMPORTS.items():
    try:
        mod = importlib.import_module(module_name)
        version = getattr(mod, "__version__", "version unknown")
        print(f"OK {label}: {version}")
    except Exception as exc:
        print(f"MISSING/ERROR {label}: {repr(exc)}")
        missing.append(label)

if missing:
    print("Missing imports:", missing)
    print("Fix: rerun the install cell, then Runtime > Restart session if Colab asks for it.")
else:
    import physicsnemo
    print("physicsnemo import ready")

## 공식 repo clone

pip 패키지는 모델 API를 제공하지만, 예제 스크립트와 `validator.py`는 GitHub repo에 있습니다. 이 셀은 공식 repo를 `/content/physicsnemo`에 받아옵니다.

In [ ]:
%cd /content
![ -d physicsnemo ] || git clone https://github.com/NVIDIA/physicsnemo.git
%cd /content/physicsnemo/examples/cfd/darcy_fno
!ls -la

## 공식 config 읽기

원본 config는 Colab 첫 실행에는 큽니다.

- `resolution=256`: 256x256 grid
- `batch_size=64`: T4/무료 Colab에는 큰 편
- `max_pseudo_epochs=256`: 공부용 smoke test에는 너무 김
- `pseudo_epoch_sample_size=2048`: pseudo epoch 하나에서 보는 샘플 수

이 노트북은 아래에서 자동으로 작은 config를 생성합니다.

In [ ]:
!sed -n '1,220p' config.yaml

## Darcy2D 데이터 이해

`Darcy2D`는 학습 중에 permeability와 Darcy solution을 계속 생성하는 datapipe입니다.

이 예제에서 물리 도메인은 `[0, 1] x [0, 1]` 정사각형 영역으로 생각하면 됩니다. 다만 노트북의 heatmap은 연속 도메인 자체가 아니라, 그 도메인을 `resolution x resolution` 격자로 샘플링한 tensor 값을 보여줍니다.

- 도메인 내부: 계산되는 격자 영역입니다. 여기에서 source term `f=1`이 고정으로 작용합니다.
- 도메인 경계: solver가 외곽에서 `u=0` Dirichlet 조건을 적용합니다. 출력 tensor는 이 조건으로 계산된 내부 grid 값을 crop해서 보여줍니다.
- 모델 입력에는 경계 조건이 따로 들어가지 않습니다. 이 예제에서는 경계 조건과 source가 항상 고정되어 있고, permeability만 바뀝니다.
- `Darcy2D`의 permeability는 자연 지층 이미지가 아니라 synthetic benchmark입니다. random Fourier field를 만든 뒤 threshold해서 보통 `k=0.5` 또는 `k=2.0` 두 값으로 보입니다.

값의 의미:

- `permeability`: PDE 계수 field입니다. 공간 위치마다 물질/매질의 투과성이 다르다고 보면 됩니다.
- `darcy`: 그 계수 field에 대해 계산된 pressure/solution field `u(x,y)`입니다.
- `normaliser`: 평균/표준편차로 값을 정규화합니다. 모델은 정규화된 값을 보고 학습합니다.

아래 셀은 실제 batch 하나를 가져와 shape, min/max/mean/std를 출력하고, 색깔이 어떤 물리값을 의미하는지 colorbar와 주석으로 보여줍니다. 이어서 `q = -k grad(u)`를 계산해 pressure field에서 실제 Darcy flux 방향도 확인합니다.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from physicsnemo.datapipes.benchmarks.darcy import Darcy2D

# GPU/CPU에 따라 데이터 크기를 자동 조정합니다.
DATA_RESOLUTION = 64 if HAS_GPU else 32
DATA_BATCH_SIZE = 4 if HAS_GPU else 1

normaliser = {
    "permeability": (1.25, 0.75),
    "darcy": (4.52e-2, 2.79e-2),
}

dataloader = Darcy2D(
    resolution=DATA_RESOLUTION,
    batch_size=DATA_BATCH_SIZE,
    normaliser=normaliser,
)

batch = next(iter(dataloader))
permeability = batch["permeability"]
darcy = batch["darcy"]

print("permeability shape:", tuple(permeability.shape))
print("darcy shape:       ", tuple(darcy.shape))

for name, tensor in [("permeability", permeability), ("darcy", darcy)]:
    print(
        name,
        "min", float(tensor.min()),
        "max", float(tensor.max()),
        "mean", float(tensor.mean()),
        "std", float(tensor.std()),
    )

permeability_raw = permeability * normaliser["permeability"][1] + normaliser["permeability"][0]
unique_k = torch.unique(torch.round(permeability_raw[0, 0].detach().cpu() * 1000) / 1000)
print("first sample unique permeability values, rounded:", unique_k.tolist())
print("Darcy2D default permeability is thresholded, so seeing mostly 0.5 and 2.0 is expected.")

In [ ]:
# 데이터 시각화: 정규화된 값을 원래 scale로 되돌려서 표시합니다.
import matplotlib.patches as patches

def denorm(x, key):
    mean, std = normaliser[key]
    return x * std + mean

p0 = denorm(permeability[0, 0].detach().cpu(), "permeability")
u0 = denorm(darcy[0, 0].detach().cpu(), "darcy")

extent = [0.0, 1.0, 0.0, 1.0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)

im0 = axes[0].imshow(p0, cmap="viridis", origin="lower", extent=extent)
axes[0].set_title("input: permeability k(x,y)")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
cb0 = plt.colorbar(im0, ax=axes[0], fraction=0.046)
cb0.set_label("permeability k: 0.5 low, 2.0 high")

im1 = axes[1].imshow(u0, cmap="magma", origin="lower", extent=extent)
axes[1].set_title("target: pressure/solution u(x,y)")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
cb1 = plt.colorbar(im1, ax=axes[1], fraction=0.046)
cb1.set_label("pressure/solution u, computed with boundary u=0")

for ax in axes:
    ax.add_patch(patches.Rectangle((0, 0), 1, 1, fill=False, edgecolor="white", linewidth=1.5))
    ax.text(0.03, 0.97, "sampled grid on [0,1]x[0,1]", color="white", fontsize=9, va="top", ha="left", bbox={"facecolor": "black", "alpha": 0.45, "pad": 3})
    ax.text(0.50, 0.08, "solver uses source f=1, boundary u=0", color="white", fontsize=9, va="bottom", ha="center", bbox={"facecolor": "black", "alpha": 0.35, "pad": 4})
    ax.set_aspect("equal")

print("이 그림은 정규화 전 원래 scale입니다.")
print("Darcy2D permeability는 random Fourier field를 threshold한 synthetic binary field입니다.")
print("k=2.0은 더 잘 흐르는 영역, k=0.5는 덜 흐르는 영역입니다. k=0인 벽은 아닙니다.")
print("solution u는 고정된 source f=1, 외곽 boundary u=0 조건에서 계산된 압력/해 field입니다.")
plt.show()

## Pressure에서 Darcy flux 보기

위 그림의 `darcy`는 유동장 자체가 아니라 scalar pressure/solution field `u(x,y)`입니다. 실제 Darcy flux는 아래처럼 후처리해서 봅니다.

```text
q(x,y) = -k(x,y) grad u(x,y)
```

화살표는 flux 방향을, 배경색은 pressure/solution 값을 뜻합니다. permeability의 보라색 영역도 `k=0.5`라서 완전히 막힌 벽이 아니며, 압력 기울기가 있으면 flux가 생깁니다.

In [ ]:
# Darcy flux q = -k grad(u) 시각화
k_np = p0.numpy()
u_np = u0.numpy()
dy, dx = 1.0 / (DATA_RESOLUTION - 1), 1.0 / (DATA_RESOLUTION - 1)

# np.gradient returns derivatives along axis 0(y) and axis 1(x).
du_dy, du_dx = np.gradient(u_np, dy, dx)
qx = -k_np * du_dx
qy = -k_np * du_dy
speed = np.sqrt(qx**2 + qy**2)

step = max(1, DATA_RESOLUTION // 16)
yy, xx = np.mgrid[0:DATA_RESOLUTION, 0:DATA_RESOLUTION]
x_coords = xx / (DATA_RESOLUTION - 1)
y_coords = yy / (DATA_RESOLUTION - 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)

im0 = axes[0].imshow(k_np, cmap="viridis", origin="lower", extent=extent)
axes[0].set_title("permeability k")
plt.colorbar(im0, ax=axes[0], fraction=0.046, label="k")

im1 = axes[1].imshow(u_np, cmap="magma", origin="lower", extent=extent)
axes[1].quiver(
    x_coords[::step, ::step],
    y_coords[::step, ::step],
    qx[::step, ::step],
    qy[::step, ::step],
    color="white",
    scale=None,
    width=0.004,
)
axes[1].set_title("pressure u + flux arrows q")
plt.colorbar(im1, ax=axes[1], fraction=0.046, label="u")

im2 = axes[2].imshow(speed, cmap="plasma", origin="lower", extent=extent)
axes[2].set_title("flux magnitude |q|")
plt.colorbar(im2, ax=axes[2], fraction=0.046, label="|q|")

for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")

print("flux q는 모델 target이 아니라, target pressure u와 permeability k로 계산한 후처리 값입니다.")
print("|q| min/max/mean:", float(speed.min()), float(speed.max()), float(speed.mean()))
plt.show()

## FNO 네트워크 구조

FNO는 격자 field 전체를 입력으로 받아 field 전체를 출력합니다.

```text
[B, 1, H, W] permeability
      -> lifting / spectral convolution blocks
      -> decoder
[B, 1, H, W] predicted Darcy solution
```

중요 config:

- `in_channels`: 입력 channel 수입니다. 지금은 permeability 하나라 `1`입니다.
- `out_channels`: 출력 channel 수입니다. Darcy solution 하나라 `1`입니다.
- `latent_channels`: 내부 feature channel 수입니다. 크면 표현력이 좋아지지만 메모리 사용량도 증가합니다.
- `num_fno_modes`: Fourier domain에서 유지할 mode 수입니다. 크면 고주파 성분을 더 잘 표현하지만 느려집니다.
- `num_fno_layers`: spectral block 반복 횟수입니다.
- `padding`: 경계 효과를 줄이기 위한 padding입니다.

In [ ]:
from physicsnemo.models.fno import FNO

MODEL_CFG = {
    "in_channels": 1,
    "out_channels": 1,
    "decoder_layers": 1,
    "decoder_layer_size": 32,
    "dimension": 2,
    "latent_channels": 16 if HAS_GPU else 8,
    "num_fno_layers": 4,
    "num_fno_modes": 8 if HAS_GPU else 4,
    "padding": 9,
}

model = FNO(**MODEL_CFG).to(DEVICE)
param_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print("MODEL_CFG:", MODEL_CFG)
print(f"parameters: {param_count:,}")
print(f"trainable:  {trainable_count:,}")

In [ ]:
# Forward pass shape check
model.eval()
with torch.no_grad():
    x = permeability.to(DEVICE)
    y = darcy.to(DEVICE)
    pred = model(x)

print("x/input shape: ", tuple(x.shape))
print("y/target shape:", tuple(y.shape))
print("pred shape:    ", tuple(pred.shape))
print("MSE before training:", torch.mean((pred - y) ** 2).item())

## 한두 step만 직접 학습하기

공식 `train_fno_darcy.py`는 로깅, checkpoint, validator까지 포함한 완성형 스크립트입니다. 공부할 때는 먼저 아래처럼 직접 training step을 보는 것이 좋습니다.

학습 한 step의 흐름:

```text
batch 생성 -> permeability를 모델에 입력 -> pred 생성 -> pred와 darcy MSE 계산 -> backward -> optimizer.step
```

In [ ]:
from torch.nn import MSELoss
from torch.optim import Adam

model.train()
loss_fun = MSELoss(reduction="mean")
optimizer = Adam(model.parameters(), lr=1e-3)

TRAIN_STEPS = 3 if HAS_GPU else 1
loss_history = []

for step, batch in zip(range(TRAIN_STEPS), dataloader):
    x = batch["permeability"].to(DEVICE)
    y = batch["darcy"].to(DEVICE)

    optimizer.zero_grad(set_to_none=True)
    pred = model(x)
    loss = loss_fun(pred, y)
    loss.backward()
    optimizer.step()

    loss_value = float(loss.detach().cpu())
    loss_history.append(loss_value)
    print(f"step {step + 1}/{TRAIN_STEPS} loss={loss_value:.6e}")

print("loss_history:", loss_history)

In [ ]:
# 예측 결과 시각화
model.eval()
with torch.no_grad():
    batch = next(iter(dataloader))
    x = batch["permeability"].to(DEVICE)
    y = batch["darcy"].to(DEVICE)
    pred = model(x)

x_img = denorm(x[0, 0].detach().cpu(), "permeability")
y_img = denorm(y[0, 0].detach().cpu(), "darcy")
pred_img = denorm(pred[0, 0].detach().cpu(), "darcy")
err_img = pred_img - y_img

fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
items = [
    (x_img, "input permeability", "viridis"),
    (y_img, "target darcy", "magma"),
    (pred_img, "prediction", "magma"),
    (err_img, "prediction - target", "coolwarm"),
]

for ax, (img, title, cmap) in zip(axes, items):
    im = ax.imshow(img, cmap=cmap)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.show()

## Colab smoke test config 생성

이 config는 공식 스크립트용입니다. GPU가 있으면 작은 GPU config를 만들고, GPU가 없으면 아주 작은 CPU fallback config를 만듭니다.

In [ ]:
from pathlib import Path

smoke_resolution = 64 if HAS_GPU else 32
smoke_batch_size = 4 if HAS_GPU else 1
smoke_latent_channels = 16 if HAS_GPU else 8
smoke_fno_modes = 8 if HAS_GPU else 4
smoke_epochs = 4 if HAS_GPU else 1
smoke_samples = 16 if HAS_GPU else 2
smoke_valid = 8 if HAS_GPU else 2

smoke_config = f"""arch:
  decoder:
    out_features: 1
    layers: 1
    layer_size: 32
  fno:
    in_channels: 1
    dimension: 2
    latent_channels: {smoke_latent_channels}
    fno_layers: 4
    fno_modes: {smoke_fno_modes}
    padding: 9
normaliser:
  permeability:
    mean: 1.25
    std_dev: .75
  darcy:
    mean: 4.52E-2
    std_dev: 2.79E-2
scheduler:
  initial_lr: 1.E-3
  decay_rate: .85
  decay_pseudo_epochs: 4
training:
  resolution: {smoke_resolution}
  batch_size: {smoke_batch_size}
  rec_results_freq : 2
  max_pseudo_epochs: {smoke_epochs}
  pseudo_epoch_sample_size: {smoke_samples}
validation:
  sample_size: {smoke_valid}
  validation_pseudo_epochs: 1
"""

Path("config_colab_smoke.yaml").write_text(smoke_config, encoding="utf-8")
print(smoke_config)

## 공식 스크립트 선택 실행

이 셀은 공식 `train_fno_darcy.py`를 실행합니다. 앞의 직접 학습 셀이 성공했다면, 모델/데이터 API는 정상입니다. 이 셀이 실패하면 대부분 로깅, checkpoint, Colab 런타임, optional dependency 문제입니다.

GPU가 없으면 이 셀은 실행하지 않고 건너뜁니다.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

RUN_OFFICIAL_SCRIPT = HAS_GPU

if not RUN_OFFICIAL_SCRIPT:
    print("Skipping official training script because GPU is not available.")
else:
    if Path("config.yaml").exists() and not Path("config_original.yaml").exists():
        shutil.copyfile("config.yaml", "config_original.yaml")

    shutil.copyfile("config_colab_smoke.yaml", "config.yaml")
    print("Running official train_fno_darcy.py with smoke config")

    result = subprocess.run([sys.executable, "train_fno_darcy.py"])
    print("return code:", result.returncode)

    if result.returncode != 0:
        print("Official script failed. The direct in-notebook data/model/training cells above are the preferred debugging path.")
        print("Common fixes: restart runtime, rerun install, check validator.py exists, reduce config, or avoid nn-extras on Colab.")

## 결과 저장

Drive가 붙어 있으면 Drive에 저장되고, 아니면 `/content` 임시 폴더에 저장됩니다.

In [ ]:
!mkdir -p "$RUN_ROOT/smoke_test"
!cp -r checkpoints "$RUN_ROOT/smoke_test/" 2>/dev/null || true
!find "$RUN_ROOT" -maxdepth 3 -type f | sed -n '1,80p'

## 커스텀 아이디어를 넣는 위치

바꾸기 쉬운 순서:

1. `MODEL_CFG`: `latent_channels`, `num_fno_modes`, `num_fno_layers`
2. `DATA_RESOLUTION`, `DATA_BATCH_SIZE`: 데이터 크기와 batch size
3. loss 함수: MSE에 gradient penalty나 relative L2 추가
4. 입력 channel: permeability 외에 coordinate grid, boundary mask 추가
5. validation metric: max error, relative error, spectrum error 추가

아래는 gradient penalty를 섞은 custom loss 예시입니다.

In [ ]:
from torch.nn import MSELoss

mse = MSELoss(reduction="mean")

def gradient_penalty_2d(pred):
    # pred: [B, C, H, W]
    dx = pred[..., 1:, :] - pred[..., :-1, :]
    dy = pred[..., :, 1:] - pred[..., :, :-1]
    return dx.square().mean() + dy.square().mean()

def custom_loss(pred, target, smooth_weight=1e-4):
    return mse(pred, target) + smooth_weight * gradient_penalty_2d(pred)

# 사용 예:
# loss = custom_loss(pred, y, smooth_weight=1e-4)

## 연구 질문 템플릿

Darcy FNO에서 바로 실험하기 좋은 질문들:

- `num_fno_modes`를 4, 8, 12, 16으로 바꾸면 고주파 error가 어떻게 변하나?
- `latent_channels`를 늘리면 loss는 줄지만 overfit이나 속도는 어떻게 변하나?
- MSE에 gradient penalty를 섞으면 prediction이 더 smooth해지나?
- `resolution=32`에서 학습하고 `resolution=64`에서 inference하면 어느 정도 유지되나?
- permeability field 분포를 바꾸면 generalization이 얼마나 무너지나?

In [ ]:
# 원본 config 복구
if Path("config_original.yaml").exists():
    shutil.copyfile("config_original.yaml", "config.yaml")
    print("Restored config_original.yaml -> config.yaml")
else:
    print("No config_original.yaml found")